# Election Prediction Project
## Notebook 03 — Model Training, Comparison and Selection

### Notebook Objectives
This notebook trains and benchmarks several models to predict the following four voting blocs:

Right_pct
Center_Left_pct
Haredi_pct
Arab_pct

### Workflow stages:
*   Chronological Split: Training on Knessets 21–24; Validation on Knesset 25.
*   Baseline Model Training: Evaluation of Linear Regression, Random Forest, and XGBoost models.
*   Feature Contribution: Evaluating the contribution of locality type variables to model performance.
*   Normalization: Normalizing predictions to ensure they sum to 100%.
*   Compositional Analysis: Training a CLR (Centered Log-Ratio) model for compositional data.
*   Segmented Modeling: Training a split CLR model based on the type_Arab/Non-Jewish category.
*   Model Finalization: Incorporating the type_Druze_Majority feature and selecting the final model.

### input

- `processed_data/modeling_dataset_selected_features.csv`
- `processed_data/selected_demographic_features.json`

### output

- `reports/model_comparison.csv`
- `reports/model_comparison_by_target.csv`
- `processed_data/validation_predictions_selected_model.csv`
- `models/final_segmented_clr_druze_bundle.joblib`
- `reports/modeling_summary.json`


## 1. Imports and project configuration

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
VALIDATION_ELECTION = "Knesset_25"

TARGET_COLUMNS = [
    "Right_pct",
    "Center_Left_pct",
    "Haredi_pct",
    "Arab_pct",
]

BASE_CLASSIFICATION_FEATURES = [
    "type_Arab/Non-Jewish",
    "type_Cities",
    "type_Kibbutzim",
    "type_Moshavim",
    "type_other",
]

DRUZE_FEATURE = "type_Druze_Majority"

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
from google.colab import drive

try:
    drive.mount("/content/drive")
except ValueError as error:
    if "already contain files" in str(error):
        print("Drive is already mounted. Continuing...")
    else:
        raise

PROJECT_DIR = Path("/content/drive/MyDrive/לימודים/פרויקט DS")
PROCESSED_DIR = PROJECT_DIR / "processed_data"
REPORTS_DIR = PROJECT_DIR / "reports"
MODELS_DIR = PROJECT_DIR / "models"

INPUT_PATH = PROCESSED_DIR / "modeling_dataset_selected_features.csv"
SELECTED_FEATURES_PATH = PROCESSED_DIR / "selected_demographic_features.json"

MODEL_COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"
TARGET_COMPARISON_PATH = REPORTS_DIR / "model_comparison_by_target.csv"
VALIDATION_PREDICTIONS_PATH = (
    PROCESSED_DIR / "validation_predictions_selected_model.csv"
)
FINAL_MODEL_BUNDLE_PATH = (
    MODELS_DIR / "final_segmented_clr_druze_bundle.joblib"
)
MODELING_SUMMARY_PATH = REPORTS_DIR / "modeling_summary.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input dataset: {INPUT_PATH}")
print(f"Selected features: {SELECTED_FEATURES_PATH}")


Mounted at /content/drive
Input dataset: /content/drive/MyDrive/לימודים/פרויקט DS/processed_data/modeling_dataset_selected_features.csv
Selected features: /content/drive/MyDrive/לימודים/פרויקט DS/processed_data/selected_demographic_features.json


## 2. Load and validate the modeling dataset

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Modeling dataset not found: {INPUT_PATH}. "
        "Run Notebook 02 first."
    )

if not SELECTED_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Selected-features file not found: {SELECTED_FEATURES_PATH}. "
        "Run Notebook 02 first."
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)

with open(SELECTED_FEATURES_PATH, "r", encoding="utf-8") as file:
    feature_payload = json.load(file)

demographic_features = feature_payload["selected_demographic_features"]

required_columns = [
    "locality_symbol",
    "year",
    "target_election",
    *TARGET_COLUMNS,
    *demographic_features,
    *BASE_CLASSIFICATION_FEATURES,
    DRUZE_FEATURE,
]

missing_columns = [
    column for column in required_columns if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["locality_symbol"] = (
    pd.to_numeric(df["locality_symbol"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")

numeric_columns = list(dict.fromkeys(
    demographic_features
    + BASE_CLASSIFICATION_FEATURES
    + [DRUZE_FEATURE]
    + TARGET_COLUMNS
))

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)

if df[TARGET_COLUMNS].isna().any().any():
    raise ValueError("Missing target values were found.")

if not np.allclose(df[TARGET_COLUMNS].sum(axis=1), 100, atol=1e-6):
    raise ValueError("Target percentages do not sum to 100%.")

if df.duplicated(
    subset=["locality_symbol", "target_election"]
).any():
    raise ValueError("Duplicate locality-election rows were found.")

print(f"Dataset shape: {df.shape}")
print(f"Selected demographic features: {len(demographic_features)}")
print("Input validation completed successfully.")

display(df.head())


Dataset shape: (1124, 41)
Selected demographic features: 25
Input validation completed successfully.


,locality_symbol,שם יישוב,year,target_election,valid_votes,Other_raw_pct,Right_pct,Center_Left_pct,Haredi_pct,Arab_pct,אחוז עולי + מסך האוכלוסייה,אחוז מקבלי השלמת הכנסה מבין מקבלי קצבאות זקנה ושאירים,אחוז זכאים לתעודת בגרות מבין תלמידי כיתות יב,עובדי הוראה ממוצע שעות עבודה שבועיות לתלמיד,מספר ילדים שבגינם שולמו קצבאות במשפחות עם 5 ילדים ויותר,השכלה גבוהה אחוז בעלי תארים מישראל מתוך אוכלוסיית בני 55-35,"מקבלי קצבאות זקנה ושאירים סה""כ (סוף שנה)","אחוז המבוטחים בקופ""ח מתוך סך כל המבוטחים כללית",avg_monthly_wage,עובדי הוראה ממוצע תלמידים למורה,אחוז באוכלוסייה בסוף השנה בני 4-0,אחוז זכאים לתעודת בגרות שעמדו בדרישות הסף של האוניברסיטאות מבין תלמידי כיתות יב,"ריבוי טבעי ל-1,000 תושבים",אחוז באוכלוסייה בסוף השנה בני 44-30,גיל ממוצע של מקבלי דמי אבטלה (לא כולל חיילים),עובדי הוראה אחוז בעלי דרגת שכר תואר שני ומעלה,שיעור פריון כולל,"יחס תלות (ל-1,000 תושבים בלתי תלויים)","מקבלי גמלת הבטחת הכנסה (נפשות, במשך השנה)",עובדי הוראה ממוצע שעות עבודה לשבוע,אחוז באוכלוסייה בסוף השנה בני 9-5,מוסלמים (אחוזים מתוך האוכלוסייה הערבית),ערבים (אחוזים מתוך כלל אוכלוסיית הישראלים),יהודים ואחרים (אחוזים מתוך כלל אוכלוסיית הישראלים),ממוצע תלמידים לכיתה בבתי ספר יסודיים,type_Arab/Non-Jewish,type_Cities,type_Kibbutzim,type_Moshavim,type_other,type_Druze_Majority
0,1015,מבשרת ציון,2019,Knesset_21,13846,2.802253,50.185763,45.623421,3.997622,0.193194,7.10,4.08,80.73,2.78,766.0,42.52,3699.0,44.9,12147.890000,10.98,7.80,70.10,10.05,18.80,42.00,45.54,2.29,901.1,229.0,30.10,7.80,93.0,0.5,99.5,24.53,0,1,0,0,0,0
1,1015,מבשרת ציון,2019,Knesset_22,13539,2.769776,45.297782,48.214828,6.236706,0.250684,7.10,4.08,80.73,2.78,766.0,42.52,3699.0,44.9,12147.890000,10.98,7.80,70.10,10.05,18.80,42.00,45.54,2.29,901.1,229.0,30.10,7.80,93.0,0.5,99.5,24.53,0,1,0,0,0,0
2,1015,מבשרת ציון,2020,Knesset_23,13559,0.678516,47.575555,45.533526,6.289448,0.601470,7.30,3.96,83.80,2.79,858.0,43.90,3862.0,45.1,13294.045859,11.17,7.87,71.34,10.72,17.88,38.12,25.68,2.55,868.0,301.0,30.50,7.54,NaN,NaN,99.7,24.18,0,1,0,0,0,0
3,1015,מבשרת ציון,2021,Knesset_24,13331,0.967669,50.780185,43.622178,5.370398,0.227238,7.39,3.97,83.13,2.92,978.0,44.97,3934.0,45.1,14557.820000,10.38,8.26,72.59,11.71,18.02,39.00,49.29,2.72,893.0,293.0,31.10,7.66,NaN,NaN,99.6,23.80,0,1,0,0,0,0
4,1015,מבשרת ציון,2022,Knesset_25,14051,1.046189,47.583429,45.792578,6.221231,0.402762,7.60,4.37,83.75,2.88,1025.0,45.74,4069.0,44.8,15201.840000,11.12,8.10,77.03,9.91,18.22,38.82,49.04,2.46,904.0,238.0,31.25,7.70,NaN,NaN,99.6,24.82,0,1,0,0,0,0


## 3. Chronological training-validation split

In [4]:
validation_mask = df["target_election"].eq(VALIDATION_ELECTION)

train_df = df.loc[~validation_mask].copy()
validation_df = df.loc[validation_mask].copy()

if train_df.empty or validation_df.empty:
    raise ValueError("Training or validation dataset is empty.")

y_train = train_df[TARGET_COLUMNS].copy()
y_validation = validation_df[TARGET_COLUMNS].copy()

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")

print("\nTraining elections:")
print(train_df["target_election"].value_counts().sort_index())

print("\nValidation elections:")
print(validation_df["target_election"].value_counts().sort_index())


Training rows: 898
Validation rows: 226

Training elections:
target_election
Knesset_21    225
Knesset_22    225
Knesset_23    224
Knesset_24    224
Name: count, dtype: int64

Validation elections:
target_election
Knesset_25    226
Name: count, dtype: int64


## 4. Reusable modeling functions



In [5]:
def prepare_feature_matrices(
    training_data,
    validation_data,
    feature_columns
):
    """Fit median imputation on training data only."""

    usable_features = [
        column
        for column in feature_columns
        if not training_data[column].isna().all()
    ]

    if not usable_features:
        raise ValueError("No usable features remain.")

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(training_data[usable_features]),
        columns=usable_features,
        index=training_data.index
    )

    X_validation = pd.DataFrame(
        imputer.transform(validation_data[usable_features]),
        columns=usable_features,
        index=validation_data.index
    )

    return {
        "X_train": X_train,
        "X_validation": X_validation,
        "imputer": imputer,
        "features": usable_features,
    }


def evaluate_predictions(y_true, predictions, model_name):
    """Return overall and target-level MAE and R2."""

    y_true_array = np.asarray(y_true, dtype=float)
    predictions = np.asarray(predictions, dtype=float)

    overall_result = {
        "Model": model_name,
        "Overall_MAE": float(
            mean_absolute_error(y_true_array, predictions)
        ),
        "Overall_R2": float(
            r2_score(y_true_array, predictions)
        ),
    }

    target_mae = mean_absolute_error(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_r2 = r2_score(
        y_true_array,
        predictions,
        multioutput="raw_values"
    )

    target_rows = [
        {
            "Model": model_name,
            "Target": target,
            "MAE": float(mae_value),
            "R2": float(r2_value),
        }
        for target, mae_value, r2_value in zip(
            TARGET_COLUMNS,
            target_mae,
            target_r2
        )
    ]

    return overall_result, target_rows


def normalize_predictions(predictions):
    """Clip negative values and normalize each row to 100%."""

    clipped = np.clip(
        np.asarray(predictions, dtype=float),
        0,
        None
    )

    row_sums = clipped.sum(axis=1, keepdims=True)

    zero_rows = row_sums.squeeze() == 0

    if zero_rows.any():
        clipped[zero_rows] = 1.0
        row_sums = clipped.sum(axis=1, keepdims=True)

    return clipped / row_sums * 100.0


def clr_transform(percentages, epsilon=1e-4):
    """Transform percentage compositions into CLR values."""

    compositions = np.asarray(percentages, dtype=float) / 100.0
    compositions = np.clip(compositions, epsilon, None)
    compositions = compositions / compositions.sum(
        axis=1,
        keepdims=True
    )

    log_values = np.log(compositions)

    return log_values - log_values.mean(
        axis=1,
        keepdims=True
    )


def inverse_clr(clr_values):
    """Convert CLR values into positive percentages summing to 100%."""

    clr_values = np.asarray(clr_values, dtype=float)
    stabilized = clr_values - clr_values.max(
        axis=1,
        keepdims=True
    )

    exponentials = np.exp(stabilized)

    return (
        exponentials
        / exponentials.sum(axis=1, keepdims=True)
        * 100.0
    )


In [6]:
def make_xgb_regressor():
    """Return the common XGBoost configuration."""

    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        n_jobs=-1,
    )


def fit_independent_model(
    training_data,
    validation_data,
    feature_columns,
    y_training,
    estimator
):
    """Train independent regressors on raw percentage targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    model = MultiOutputRegressor(clone(estimator))

    model.fit(prepared["X_train"], y_training)

    predictions = model.predict(
        prepared["X_validation"]
    )

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
    }


def fit_clr_model(
    training_data,
    validation_data,
    feature_columns,
    y_training
):
    """Train XGBoost models on CLR-transformed targets."""

    prepared = prepare_feature_matrices(
        training_data,
        validation_data,
        feature_columns
    )

    y_training_clr = clr_transform(y_training)

    model = MultiOutputRegressor(
        make_xgb_regressor()
    )

    model.fit(
        prepared["X_train"],
        y_training_clr
    )

    predicted_clr = model.predict(
        prepared["X_validation"]
    )

    predictions = inverse_clr(predicted_clr)

    return {
        **prepared,
        "model": model,
        "predictions": predictions,
        "predicted_clr": predicted_clr,
    }


def fit_segmented_clr_model(
    training_data,
    validation_data,
    arab_feature_columns,
    non_arab_feature_columns
):
    """Train separate CLR models for Arab and non-Arab localities."""

    group_column = "type_Arab/Non-Jewish"

    arab_train = training_data.loc[
        training_data[group_column].eq(1)
    ].copy()

    arab_validation = validation_data.loc[
        validation_data[group_column].eq(1)
    ].copy()

    non_arab_train = training_data.loc[
        training_data[group_column].eq(0)
    ].copy()

    non_arab_validation = validation_data.loc[
        validation_data[group_column].eq(0)
    ].copy()

    for name, subset in {
        "Arab training": arab_train,
        "Arab validation": arab_validation,
        "Non-Arab training": non_arab_train,
        "Non-Arab validation": non_arab_validation,
    }.items():
        if subset.empty:
            raise ValueError(f"{name} subset is empty.")

    arab_result = fit_clr_model(
        arab_train,
        arab_validation,
        arab_feature_columns,
        arab_train[TARGET_COLUMNS]
    )

    non_arab_result = fit_clr_model(
        non_arab_train,
        non_arab_validation,
        non_arab_feature_columns,
        non_arab_train[TARGET_COLUMNS]
    )

    predictions_df = pd.DataFrame(
        index=validation_data.index,
        columns=TARGET_COLUMNS,
        dtype=float
    )

    predictions_df.loc[
        arab_validation.index,
        TARGET_COLUMNS
    ] = arab_result["predictions"]

    predictions_df.loc[
        non_arab_validation.index,
        TARGET_COLUMNS
    ] = non_arab_result["predictions"]

    if predictions_df.isna().any().any():
        raise ValueError(
            "Some validation rows did not receive predictions."
        )

    return {
        "predictions": (
            predictions_df
            .loc[validation_data.index]
            .to_numpy(dtype=float)
        ),
        "arab_result": arab_result,
        "non_arab_result": non_arab_result,
        "arab_train_rows": len(arab_train),
        "arab_validation_rows": len(arab_validation),
        "non_arab_train_rows": len(non_arab_train),
        "non_arab_validation_rows": len(non_arab_validation),
    }


## 5. Baseline models using demographic features only

In [7]:
overall_results = []
target_results = []
prediction_store = {}

baseline_estimators = {
    "Linear Regression — Demographic": LinearRegression(),
    "Random Forest — Demographic": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost — Demographic": make_xgb_regressor(),
}

baseline_models = {}

for model_name, estimator in baseline_estimators.items():
    print(f"Training: {model_name}")

    result = fit_independent_model(
        train_df,
        validation_df,
        demographic_features,
        y_train,
        estimator
    )

    baseline_models[model_name] = result
    prediction_store[model_name] = result["predictions"]

    overall_row, target_rows = evaluate_predictions(
        y_validation,
        result["predictions"],
        model_name
    )

    overall_results.append(overall_row)
    target_results.extend(target_rows)

    print(
        f"MAE: {overall_row['Overall_MAE']:.3f} | "
        f"R2: {overall_row['Overall_R2']:.3f}"
    )

baseline_comparison = pd.DataFrame(
    overall_results
).sort_values("Overall_MAE")

display(baseline_comparison.round(3))


Training: Linear Regression — Demographic
MAE: 14.194 | R2: 0.563
Training: Random Forest — Demographic
MAE: 12.907 | R2: 0.357
Training: XGBoost — Demographic
MAE: 11.459 | R2: 0.516


,Model,Overall_MAE,Overall_R2
2,XGBoost — Demographic,11.459,0.516
1,Random Forest — Demographic,12.907,0.357
0,Linear Regression — Demographic,14.194,0.563


## 6. XGBoost with locality-type classifications

In [8]:
full_classification_features = list(
    dict.fromkeys(
        demographic_features
        + BASE_CLASSIFICATION_FEATURES
    )
)

classification_model_name = (
    "XGBoost — Demographic + Locality Types"
)

classification_result = fit_independent_model(
    train_df,
    validation_df,
    full_classification_features,
    y_train,
    make_xgb_regressor()
)

raw_classification_predictions = (
    classification_result["predictions"]
)

prediction_store[
    classification_model_name
] = raw_classification_predictions

overall_row, target_rows = evaluate_predictions(
    y_validation,
    raw_classification_predictions,
    classification_model_name
)

overall_results.append(overall_row)
target_results.extend(target_rows)

print(
    f"MAE: {overall_row['Overall_MAE']:.3f}\n"
    f"R2:  {overall_row['Overall_R2']:.3f}"
)


MAE: 8.238
R2:  0.740


## 7. Post-hoc normalization



In [9]:
normalized_model_name = (
    "XGBoost + Locality Types — Post-hoc Normalized"
)

normalized_predictions = normalize_predictions(
    raw_classification_predictions
)

prediction_store[
    normalized_model_name
] = normalized_predictions

overall_row, target_rows = evaluate_predictions(
    y_validation,
    normalized_predictions,
    normalized_model_name
)

overall_results.append(overall_row)
target_results.extend(target_rows)

normalized_sums = normalized_predictions.sum(axis=1)

print(f"Minimum sum: {normalized_sums.min():.6f}")
print(f"Maximum sum: {normalized_sums.max():.6f}")
print(f"MAE: {overall_row['Overall_MAE']:.3f}")
print(f"R2:  {overall_row['Overall_R2']:.3f}")


Minimum sum: 100.000000
Maximum sum: 100.000000
MAE: 8.183
R2:  0.768


## 8. CLR composition model

במודל זה XGBoost מאומן על טרנספורמציית CLR. פונקציית ה־Loss היא Squared Error במרחב CLR, ולא באחוזים המקוריים.


In [ ]:
clr_model_name = (
    "CLR XGBoost — Demographic + Locality Types"
)

clr_result = fit_clr_model(
    train_df,
    validation_df,
    full_classification_features,
    y_train
)

clr_predictions = clr_result["predictions"]

prediction_store[clr_model_name] = clr_predictions

overall_row, target_rows = evaluate_predictions(
    y_validation,
    clr_predictions,
    clr_model_name
)

overall_results.append(overall_row)
target_results.extend(target_rows)

clr_sums = clr_predictions.sum(axis=1)

print(f"Minimum sum: {clr_sums.min():.6f}")
print(f"Maximum sum: {clr_sums.max():.6f}")
print(f"MAE: {overall_row['Overall_MAE']:.3f}")
print(f"R2:  {overall_row['Overall_R2']:.3f}")


## 9. Segmented CLR model

Two models are trained: one for Arab/Non-Jewish localities and another for the rest.

This split allows each model to learn more homogeneous voting patterns.


In [10]:
arab_features_without_druze = demographic_features.copy()

non_arab_classification_features = [
    column
    for column in BASE_CLASSIFICATION_FEATURES
    if column != "type_Arab/Non-Jewish"
]

non_arab_features = list(
    dict.fromkeys(
        demographic_features
        + non_arab_classification_features
    )
)

segmented_model_name = "Segmented CLR"

segmented_result = fit_segmented_clr_model(
    train_df,
    validation_df,
    arab_features_without_druze,
    non_arab_features
)

segmented_predictions = segmented_result["predictions"]

prediction_store[
    segmented_model_name
] = segmented_predictions

overall_row, target_rows = evaluate_predictions(
    y_validation,
    segmented_predictions,
    segmented_model_name
)

overall_results.append(overall_row)
target_results.extend(target_rows)

print(
    f"Arab train rows: "
    f"{segmented_result['arab_train_rows']}"
)
print(
    f"Arab validation rows: "
    f"{segmented_result['arab_validation_rows']}"
)
print(
    f"Non-Arab train rows: "
    f"{segmented_result['non_arab_train_rows']}"
)
print(
    f"Non-Arab validation rows: "
    f"{segmented_result['non_arab_validation_rows']}"
)
print(f"MAE: {overall_row['Overall_MAE']:.3f}")
print(f"R2:  {overall_row['Overall_R2']:.3f}")


Arab train rows: 324
Arab validation rows: 81
Non-Arab train rows: 574
Non-Arab validation rows: 145
MAE: 4.133
R2:  0.929


## 10. Final model: Segmented CLR with Druze indicator


In [11]:
arab_features_with_druze = list(
    dict.fromkeys(
        demographic_features
        + [DRUZE_FEATURE]
    )
)

final_model_name = (
    "Segmented CLR + Druze Indicator"
)

final_result = fit_segmented_clr_model(
    train_df,
    validation_df,
    arab_features_with_druze,
    non_arab_features
)

final_predictions = final_result["predictions"]

prediction_store[
    final_model_name
] = final_predictions

overall_row, target_rows = evaluate_predictions(
    y_validation,
    final_predictions,
    final_model_name
)

overall_results.append(overall_row)
target_results.extend(target_rows)

final_sums = final_predictions.sum(axis=1)

print(f"Minimum sum: {final_sums.min():.6f}")
print(f"Maximum sum: {final_sums.max():.6f}")
print(f"Minimum prediction: {final_predictions.min():.6f}")
print(f"Maximum prediction: {final_predictions.max():.6f}")
print(f"MAE: {overall_row['Overall_MAE']:.3f}")
print(f"R2:  {overall_row['Overall_R2']:.3f}")


Minimum sum: 100.000000
Maximum sum: 100.000000
Minimum prediction: 0.007389
Maximum prediction: 96.735189
MAE: 3.992
R2:  0.931


## 11. Compare all models

In [12]:
model_comparison_df = (
    pd.DataFrame(overall_results)
    .drop_duplicates(subset="Model", keep="last")
    .sort_values("Overall_MAE")
    .reset_index(drop=True)
)

target_comparison_df = (
    pd.DataFrame(target_results)
    .drop_duplicates(
        subset=["Model", "Target"],
        keep="last"
    )
)

display(model_comparison_df.round(3))

print("\nSelected model results by target:")

display(
    target_comparison_df.loc[
        target_comparison_df["Model"].eq(
            final_model_name
        )
    ]
    .round(3)
    .reset_index(drop=True)
)

best_model_row = model_comparison_df.iloc[0]

print("\nBest validation model by MAE:")
print(f"Model: {best_model_row['Model']}")
print(f"MAE: {best_model_row['Overall_MAE']:.3f}")
print(f"R2:  {best_model_row['Overall_R2']:.3f}")


,Model,Overall_MAE,Overall_R2
0,Segmented CLR + Druze Indicator,3.992,0.931
1,Segmented CLR,4.133,0.929
2,XGBoost + Locality Types — Post-hoc Normalized,8.183,0.768
3,XGBoost — Demographic + Locality Types,8.238,0.740
4,XGBoost — Demographic,11.459,0.516
5,Random Forest — Demographic,12.907,0.357
6,Linear Regression — Demographic,14.194,0.563



Selected model results by target:


,Model,Target,MAE,R2
0,Segmented CLR + Druze Indicator,Right_pct,5.763,0.899
1,Segmented CLR + Druze Indicator,Center_Left_pct,5.212,0.892
2,Segmented CLR + Druze Indicator,Haredi_pct,2.013,0.955
3,Segmented CLR + Druze Indicator,Arab_pct,2.977,0.978



Best validation model by MAE:
Model: Segmented CLR + Druze Indicator
MAE: 3.992
R2:  0.931


## 12. Save predictions, comparison tables and final model bundle

In [13]:
metadata_columns = [
    column
    for column in [
        "locality_symbol",
        "שם יישוב",
        "year",
        "target_election",
        "valid_votes",
        *BASE_CLASSIFICATION_FEATURES,
        DRUZE_FEATURE,
    ]
    if column in validation_df.columns
]

validation_predictions_df = (
    validation_df[
        metadata_columns + TARGET_COLUMNS
    ]
    .reset_index(drop=True)
    .copy()
)

for target_index, target in enumerate(TARGET_COLUMNS):
    predicted_column = f"Predicted_{target}"
    error_column = f"Absolute_Error_{target}"

    validation_predictions_df[
        predicted_column
    ] = final_predictions[:, target_index]

    validation_predictions_df[
        error_column
    ] = np.abs(
        validation_predictions_df[target]
        - validation_predictions_df[
            predicted_column
        ]
    )

validation_predictions_df[
    "Mean_Absolute_Error"
] = (
    validation_predictions_df[
        [
            f"Absolute_Error_{target}"
            for target in TARGET_COLUMNS
        ]
    ]
    .mean(axis=1)
)

validation_predictions_df[
    "Predicted_Total"
] = (
    validation_predictions_df[
        [
            f"Predicted_{target}"
            for target in TARGET_COLUMNS
        ]
    ]
    .sum(axis=1)
)

display(
    validation_predictions_df
    .sort_values(
        "Mean_Absolute_Error",
        ascending=False
    )
    .head(10)
    .round(3)
)


,locality_symbol,שם יישוב,year,target_election,valid_votes,type_Arab/Non-Jewish,type_Cities,type_Kibbutzim,type_Moshavim,type_other,type_Druze_Majority,Right_pct,Center_Left_pct,Haredi_pct,Arab_pct,Predicted_Right_pct,Absolute_Error_Right_pct,Predicted_Center_Left_pct,Absolute_Error_Center_Left_pct,Predicted_Haredi_pct,Absolute_Error_Haredi_pct,Predicted_Arab_pct,Absolute_Error_Arab_pct,Mean_Absolute_Error,Predicted_Total
73,33,בת שלמה,2022,Knesset_25,323,0,0,0,1,0,0,26.752,71.656,0.955,0.637,62.304,35.553,16.987,54.669,20.666,19.711,0.042,0.595,27.632,100.0
27,13,חצבה,2022,Knesset_25,407,0,0,0,1,0,0,14.286,83.208,1.253,1.253,58.207,43.921,37.408,45.800,4.369,3.116,0.016,1.237,23.518,100.0
103,4501,ע'ג'ר,2022,Knesset_25,547,1,0,0,0,0,0,17.769,50.095,13.422,18.715,24.739,6.970,17.229,32.866,1.012,12.409,57.020,38.305,22.638,100.0
26,1296,כסרא-סמיע,2022,Knesset_25,1945,1,0,0,0,0,1,30.553,62.044,0.000,7.404,57.460,26.907,33.755,28.289,2.838,2.838,5.947,1.456,14.872,100.0
219,962,טובא-זנגרייה,2022,Knesset_25,2083,1,0,0,0,0,0,4.396,5.845,0.531,89.227,20.215,15.819,18.662,12.817,0.636,0.105,60.487,28.740,14.370,100.0
94,3826,שער שומרון,2022,Knesset_25,4162,0,1,0,0,0,0,74.004,21.842,4.105,0.049,46.455,27.550,47.950,26.108,5.422,1.318,0.173,0.124,13.775,100.0
113,480,בית ג'ן,2022,Knesset_25,4832,1,0,0,0,0,1,14.848,81.780,0.105,3.267,41.271,26.423,54.408,27.372,0.749,0.645,3.571,0.304,13.686,100.0
39,18,שדה משה,2022,Knesset_25,322,0,0,0,1,0,0,37.580,59.554,2.866,0.000,63.200,25.621,33.216,26.338,3.558,0.692,0.025,0.025,13.169,100.0
141,525,סאג'ור,2022,Knesset_25,1704,1,0,0,0,0,1,70.346,27.246,0.763,1.644,44.238,26.108,44.500,17.254,8.181,7.418,3.081,1.436,13.054,100.0
123,496,חורפיש,2022,Knesset_25,2577,1,0,0,0,0,1,19.061,74.191,2.723,4.025,37.653,18.592,49.014,25.177,4.634,1.911,8.699,4.674,12.589,100.0


In [14]:
final_model_bundle = {
    "model_name": final_model_name,
    "validation_election": VALIDATION_ELECTION,
    "target_columns": TARGET_COLUMNS,
    "group_column": "type_Arab/Non-Jewish",
    "druze_feature": DRUZE_FEATURE,
    "arab_model": final_result[
        "arab_result"
    ]["model"],
    "arab_imputer": final_result[
        "arab_result"
    ]["imputer"],
    "arab_features": final_result[
        "arab_result"
    ]["features"],
    "non_arab_model": final_result[
        "non_arab_result"
    ]["model"],
    "non_arab_imputer": final_result[
        "non_arab_result"
    ]["imputer"],
    "non_arab_features": final_result[
        "non_arab_result"
    ]["features"],
    "clr_epsilon": 1e-4,
    "xgboost_parameters": {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_STATE,
        "objective": "reg:squarederror",
    },
}

joblib.dump(
    final_model_bundle,
    FINAL_MODEL_BUNDLE_PATH
)

model_comparison_df.to_csv(
    MODEL_COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig"
)

target_comparison_df.to_csv(
    TARGET_COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig"
)

validation_predictions_df.to_csv(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Artifacts saved:")
print(f"- {FINAL_MODEL_BUNDLE_PATH}")
print(f"- {MODEL_COMPARISON_PATH}")
print(f"- {TARGET_COMPARISON_PATH}")
print(f"- {VALIDATION_PREDICTIONS_PATH}")


Artifacts saved:
- /content/drive/MyDrive/לימודים/פרויקט DS/models/final_segmented_clr_druze_bundle.joblib
- /content/drive/MyDrive/לימודים/פרויקט DS/reports/model_comparison.csv
- /content/drive/MyDrive/לימודים/פרויקט DS/reports/model_comparison_by_target.csv
- /content/drive/MyDrive/לימודים/פרויקט DS/processed_data/validation_predictions_selected_model.csv


In [15]:
selected_overall = (
    model_comparison_df.loc[
        model_comparison_df["Model"].eq(
            final_model_name
        )
    ]
    .iloc[0]
)

selected_by_target = (
    target_comparison_df.loc[
        target_comparison_df["Model"].eq(
            final_model_name
        )
    ]
    .set_index("Target")
)

modeling_summary = {
    "selected_model": final_model_name,
    "validation_election": VALIDATION_ELECTION,
    "training_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "demographic_feature_count": int(
        len(demographic_features)
    ),
    "arab_model_feature_count": int(
        len(final_model_bundle["arab_features"])
    ),
    "non_arab_model_feature_count": int(
        len(final_model_bundle["non_arab_features"])
    ),
    "overall_mae": float(
        selected_overall["Overall_MAE"]
    ),
    "overall_r2": float(
        selected_overall["Overall_R2"]
    ),
    "target_results": {
        target: {
            "mae": float(
                selected_by_target.loc[target, "MAE"]
            ),
            "r2": float(
                selected_by_target.loc[target, "R2"]
            ),
        }
        for target in TARGET_COLUMNS
    },
    "loss_function": (
        "Squared Error in CLR space"
    ),
    "prediction_constraint": (
        "Positive percentages summing to 100"
    ),
}

with open(
    MODELING_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        modeling_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Modeling summary saved: {MODELING_SUMMARY_PATH}")


Modeling summary saved: /content/drive/MyDrive/לימודים/פרויקט DS/reports/modeling_summary.json


## 13. Notebook summary

In this notebook, we compared linear models, tree-based models, and boosting algorithms. Switching to a CLR (Centered Log-Ratio) model resolved the summation constraint on predictions, while segmenting by locality type significantly enhanced performance. The inclusion of the type_Druze_Majority feature yielded our final model: Segmented CLR + Druze Indicator.

In the next notebook, we will perform:
*   Performance Metrics: Detailed analysis of MAE and R².
*   Visualization: Actual vs. Predicted plots.
*   Benchmarking: Performance comparison by voting bloc.
*   Error Analysis: Breakdown of errors by locality type.
*   Deep Dive: Identifying the top 10 localities with the highest prediction errors.
*   Discussion: Reviewing model limitations and future work.


